# Azure OpenAI API Call Test

A step-by-step notebook for making your first API call to an **Azure OpenAI** deployment,
reading the response back, and evaluating it with a simple quality metric.

**What's inside**
1. Install the required packages
2. Set your Azure OpenAI credentials
3. Make an API call and print the response
4. Inspect the full response object (token usage, finish reason)
5. Try your own prompt
6. LLM-as-a-Judge — QA Relevancy Metric (score how well the answer addressed the question)

**Prerequisites**
- Python 3.10+ (Google Colab works out of the box)
- An Azure OpenAI resource with a deployment ready to use:
  - **Endpoint** — the URL of your Azure OpenAI resource
  - **API key** — your secret key (keep this private)
  - **Deployment name** — the name you gave the model when you deployed it
  - **API version** — the Azure OpenAI REST API version string

> Just run the cells top to bottom. Each section has a short explanation before the code.

## 1. Install dependencies

Run this once per session. It installs the `openai` Python package, which is the official
library for talking to Azure OpenAI (and regular OpenAI) from Python.

If the packages are already installed you can skip this cell.

In [ ]:
%pip install \
    openai==1.99.5 \
    langchain-openai==0.3.28
print('Done. Skip this cell next time if packages are already installed.')

## 2. Set your Azure OpenAI credentials

Fill in the four values below with your own Azure OpenAI details.

| Variable | What to put here |
|---|---|
| `AZURE_OPENAI_API_KEY` | Your secret API key from the Azure portal |
| `AZURE_OPENAI_API_VERSION` | e.g. `"2025-04-01-preview"` |
| `AZURE_OPENAI_ENDPOINT` | e.g. `"https://your-resource-name.openai.azure.com/"` |
| `AZURE_DEPLOYMENT_NAME` | The name of your model deployment, e.g. `"gpt-5"` |

> **Keep your API key private.** Never share a notebook with your real key inside it.
> A good practice is to copy this notebook, fill in the key only in your private copy,
> and share the template (with the placeholder) with others.

In [ ]:
import os

# 👇 Replace these with your Azure OpenAI values
os.environ["AZURE_OPENAI_API_KEY"]     = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_API_VERSION"] = "YOUR_AZURE_OPENAI_API_VERSION"
os.environ["AZURE_OPENAI_ENDPOINT"]    = "YOUR_AZURE_OPENAI_ENDPOINT"
os.environ["AZURE_DEPLOYMENT_NAME"]    = "YOUR_AZURE_DEPLOYMENT_NAME"

In [ ]:
# Confirm the values were picked up (the key itself is never printed).
api_key     = os.environ.get('AZURE_OPENAI_API_KEY')
api_version = os.environ.get('AZURE_OPENAI_API_VERSION')
endpoint    = os.environ.get('AZURE_OPENAI_ENDPOINT')
deployment  = os.environ.get('AZURE_DEPLOYMENT_NAME')

print('API key present?', bool(api_key))
print('API version:    ', api_version)
print('Endpoint:       ', endpoint)
print('Deployment:     ', deployment)

## 3. Make an API call and read the response

This is the core of the notebook. Here is what happens step by step:

1. **Create a client** — an `AzureOpenAI` object that knows your endpoint, key, and
   API version.
2. **Send a message** — call `client.chat.completions.create(...)` with your prompt
   inside a `messages` list.
3. **Read the reply** — the model's text lives at
   `response.choices[0].message.content`.

### The `messages` list

Every conversation with the model is structured as a list of messages, each with a
`role` and some `content`:

| Role | Purpose |
|---|---|
| `system` | Sets the model's behaviour and persona for the whole conversation |
| `user` | Your question or instruction |
| `assistant` | The model's reply (only needed when building multi-turn conversations) |

### GPT-5 and `temperature`

GPT-5 (and the o-series reasoning models) only accept **`temperature = 1`** — any
other value is rejected by the API with an error. The cell below sets it explicitly
so the call works regardless of the model's default.

In [ ]:
from openai import AzureOpenAI

# Step 1 — create the client.
# This object is reusable: create it once and call it as many times as you like.
client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

# Step 2 — send a message.
# Edit the content strings below to change the system instruction or your question.
response = client.chat.completions.create(
    model=deployment,           # the deployment name you set above
    temperature=1,              # GPT-5 requires exactly 1; safe default for all models
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer clearly and concisely.",
        },
        {
            "role": "user",
            "content": "What is Azure OpenAI, and how is it different from regular OpenAI?",
        },
    ],
)

# Step 3 — read and print the reply.
reply = response.choices[0].message.content
print('Model reply:')
print('─' * 60)
print(reply)
print('─' * 60)

## 4. Inspect the full response object (optional)

The `response` object returned by the API contains more than just the text — it also
carries metadata like token usage and the model's finish reason. Printing it here helps
you understand the full shape of what the API returns.

| Field | What it tells you |
|---|---|
| `response.model` | The exact model version that handled the request |
| `response.usage.prompt_tokens` | How many tokens your input used |
| `response.usage.completion_tokens` | How many tokens the model's reply used |
| `response.choices[0].finish_reason` | Why the model stopped (`stop`, `length`, etc.) |

In [ ]:
print('Model used:      ', response.model)
print('Finish reason:   ', response.choices[0].finish_reason)
print()
print('Token usage:')
print('  Prompt tokens:    ', response.usage.prompt_tokens)
print('  Completion tokens:', response.usage.completion_tokens)
print('  Total tokens:     ', response.usage.total_tokens)

## 5. Try your own prompt

Change the `system_message` and `user_message` strings below and run the cell to test
any question you like. Everything else (the client, credentials, temperature) stays
the same — you only need to touch these two strings.

In [ ]:
# ✏️ Edit these two strings to ask your own question.
system_message = "You are a helpful assistant."
user_message   = "Explain what a REST API is in two sentences."

custom_response = client.chat.completions.create(
    model=deployment,
    temperature=1,
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message},
    ],
)

print('Your question:')
print(' ', user_message)
print()
print('Model reply:')
print('─' * 60)
print(custom_response.choices[0].message.content)
print('─' * 60)

## 6. LLM-as-a-Judge — QA Relevancy Metric

Now that you know how to call an LLM and get an answer, the natural next question is:
**how good was that answer?**

This section introduces a simple evaluation metric called **QA Relevancy** — short for
*Question-Answer Relevancy*. The idea is straightforward:

1. You ask the LLM a **question** and get its **answer** (exactly like Section 3).
2. A second call — the **judge** — looks at the same question and the answer the LLM
   gave, and scores how well the answer actually addresses the question.

> The judge is just another call to the same model with a carefully written prompt.
> That is the essence of the *LLM-as-a-Judge* pattern.

### What the score means

| Score | Meaning |
|---|---|
| **5** | The answer is completely on-topic and directly addresses the question |
| **4** | Mostly relevant, with minor tangents |
| **3** | Partially relevant — addresses the question but misses key parts |
| **2** | Loosely related but largely misses the point |
| **1** | Irrelevant — the answer does not address the question at all |

### The judge prompt

The prompt is stored in `QA_RELEVANCY_PROMPT` so you can edit it in one place without
touching the rest of the code. The model is instructed to respond with a small JSON
object containing a numeric `score` and a `reason`. If you change the scoring scale
or add extra fields, update the parsing cell below to match.

In [ ]:
# ✏️ This is the judge prompt. Edit it here to change how scoring works.
# The model MUST return a JSON object with at least a "score" key.
# You can add extra keys (e.g. "confidence") — just update the print block in 6.2.

QA_RELEVANCY_PROMPT = """You are a rigorous and impartial QA Relevancy evaluator. Your sole job is to \
assess how well an AI assistant's answer addresses the specific question that was asked.

## Definition of QA Relevancy
Relevancy measures whether the answer is directed at the actual question asked — \
not whether the answer is factually correct, well-written, or detailed. \
A beautifully written answer about the wrong topic scores 1. \
A brief but fully on-point answer scores 5.

## Your evaluation process (follow this order)
Step 1 — Identify the core intent of the question: what is the user actually trying to learn or accomplish?
Step 2 — Identify what the answer actually covers: what topic or topics does it address?
Step 3 — Compare: does the answer's coverage map onto the question's intent?
  - Does it address ALL parts of the question, or only some?
  - Does it introduce content that has no bearing on the question?
  - Does it stay focused, or drift into tangential detail?
Step 4 — Assign a score using the rubric below.
Step 5 — Write a single, specific sentence explaining your score. Reference actual content from the question and answer.

## Scoring rubric
5 — Fully relevant. The answer directly and completely addresses the question. Every sentence contributes. No off-topic content.
4 — Mostly relevant. The answer clearly addresses the question but contains one minor tangent or omits a small sub-part of the question.
3 — Partially relevant. The answer addresses the general topic but misses at least one significant part of the question, OR includes a notable off-topic section.
2 — Marginally relevant. The answer touches on a related topic but does not substantively answer what was asked. A reader would still need to look elsewhere.
1 — Not relevant. The answer addresses a different question entirely, or is so off-topic that it provides no useful response to what was asked.

## Critical rules
- Score based ONLY on relevance to the question — ignore grammar, tone, factual accuracy, and length.
- Do not reward verbosity. A long answer that buries the point is not more relevant than a short, direct one.
- Do not penalise brevity. A concise, focused answer that fully addresses the question scores 5.
- Be strict at the boundaries: if uncertain between two scores, ask whether the answer would leave the user satisfied — if not, take the lower score.

## Output format
Respond with ONLY a valid JSON object. No markdown, no code fences, no commentary outside the JSON.

{
  "reasoning": "<two or three sentences: what the question asks, what the answer covers, and where they match or diverge>",
  "score": <integer 1-5>,
  "reason": "<one precise sentence explaining the score, referencing specific content>"
}
"""

print('Judge prompt loaded. Edit QA_RELEVANCY_PROMPT above to change scoring behaviour.')

### 6.1 Ask the LLM a question and capture the answer

Set your question below. The LLM will answer it, and that answer becomes the input to
the QA Relevancy judge in the next cell.

In [ ]:
# ✏️ Change this question to whatever you want to test.
qa_question = (
    "I have been feeling very tired and low on energy for the past few weeks even "
    "though I am sleeping enough. What could be causing this and what should I do?"
)

# Call the LLM to get an answer — same pattern as Section 3.
qa_response = client.chat.completions.create(
    model=deployment,
    temperature=1,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a knowledgeable and empathetic general assistant. "
                "Answer questions clearly, helpfully, and in plain language. "
                "When appropriate, suggest practical next steps the person can take."
            ),
        },
        {
            "role": "user",
            "content": qa_question,
        },
    ],
)

qa_answer = qa_response.choices[0].message.content

print('Question:')
print(qa_question)
print()
print('LLM Answer:')
print('─' * 60)
print(qa_answer)
print('─' * 60)

### 6.2 Run the QA Relevancy judge

The judge receives the question from above and the answer the LLM just gave, then
scores how well the answer addressed the question. The score and the reason are parsed
out of the model's JSON response and printed in a readable format.

In [ ]:
import json

# Build the user message for the judge: plug the question and answer into the prompt.
judge_user_message = (
    f"QUESTION:\n{qa_question}\n\n"
    f"ANSWER:\n{qa_answer}"
)

# Call the judge — a second LLM call that evaluates the first one's output.
judge_response = client.chat.completions.create(
    model=deployment,
    temperature=1,
    messages=[
        {"role": "system", "content": QA_RELEVANCY_PROMPT},
        {"role": "user",   "content": judge_user_message},
    ],
)

raw_judge_output = judge_response.choices[0].message.content.strip()

# Parse the JSON the judge returned.
def parse_judge_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Strip markdown code fences if the model added them despite instructions.
        cleaned = text.strip("` \n").removeprefix("json").strip()
        return json.loads(cleaned)   # raises if still unparseable

try:
    result    = parse_judge_json(raw_judge_output)
    score     = result.get("score",     "N/A")
    reason    = result.get("reason",    "No reason provided.")
    reasoning = result.get("reasoning", "")
except json.JSONDecodeError:
    score     = "parse error"
    reason    = "Could not parse the judge's response as JSON."
    reasoning = f"Raw output was:\n{raw_judge_output}"

# Display the result.
print('QA Relevancy Metric Result')
print('═' * 60)
print(f'Score    : {score} / 5')
print()
if reasoning:
    print(f'Reasoning: {reasoning}')
    print()
print(f'Verdict  : {reason}')
print('═' * 60)

## Recap

You have successfully:
- Installed the `openai` SDK and pointed it at an **Azure** endpoint
- Created an `AzureOpenAI` client with your credentials
- Sent a `chat.completions.create` request with a system instruction and a user question
- Read the model's reply and inspected the token usage metadata
- Used a **second LLM call as a judge** to score the QA Relevancy of the first answer

**Where to go next**
- Add more `{"role": "user"}` / `{"role": "assistant"}` turns to the `messages` list
  to build a multi-turn conversation.
- Adjust `max_tokens` in the `create` call to control how long the reply can be.
- Swap the `deployment` value to test a different Azure model deployment.
- Edit `QA_RELEVANCY_PROMPT` to change the scoring criteria or output format.
- Add more judge metrics (e.g. Faithfulness, Conciseness) by writing additional prompts
  that follow the same pattern as Section 6.

📚 Azure OpenAI docs: https://learn.microsoft.com/azure/ai-services/openai/  
📚 OpenAI Python SDK: https://github.com/openai/openai-python